In [ ]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

In [ ]:
# 2. Get a fresh copy of main
%cd /content
!rm -rf kv-evidence-loss
!git clone -q https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

In [ ]:
# 3. Install the pinned team environment
# kvpress 0.5.5 needs transformers<5.3 and pandas<3; torch is Colab's own CUDA build
!pip install -q -r requirements.txt
!pip install -q kvpress==0.5.5 transformers==5.2.0 pandas==2.2.3
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|kvpress|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

In [ ]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, every package)
!mkdir -p results/answers
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; pip freeze) > results/answers/full_1.0_env.txt
!head -4 results/answers/full_1.0_env.txt

In [ ]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

In [ ]:
# 6. Keep the old baseline for comparison, then run (first run downloads Qwen3-8B, ~16 GB)
!cp results/answers/full_1.0.parquet /content/full_1.0_old.parquet
!python scripts/run_full_baseline.py

In [ ]:
# 7. Compare with the old baseline
import pandas as pd
old_df = pd.read_parquet("/content/full_1.0_old.parquet")
new_df = pd.read_parquet("results/answers/full_1.0.parquet")
if "id" in old_df.columns:
    old_df = old_df.set_index("id")
    new_df = new_df.set_index("id")
both_df = old_df.join(new_df, lsuffix="_old", rsuffix="_new")
changed_filt = both_df["answer_old"] != both_df["answer_new"]
print("questions:", len(both_df))
print("changed answers:", changed_filt.sum())
for col in ["contains", "contains_alias", "f1"]:
    print(col, round(old_df[col].mean(), 3), "->", round(new_df[col].mean(), 3))
both_df.loc[changed_filt, ["answer_old", "answer_new"]]

In [ ]:
# 8. Download the results and the environment record
from google.colab import files
files.download("results/answers/full_1.0.parquet")
files.download("results/answers/full_1.0_check.csv")
files.download("results/answers/full_1.0_env.txt")